In [1]:
import xml.etree.ElementTree as ET
import pandas as pd
from collections import defaultdict
import requests
from time import sleep
import os
import re
import pdfplumber

from datetime import datetime

import sys
sys.path.append("..")
from utils.countrymerger import loadKeyDf

In [2]:
SAVEPATH = "../../data/raw/unhumanrights"

In [475]:
#TODO
#Достать Related Document: Draft resolution E/CN.4/1991/L.22 даже там, где нет прямой ссылки
#Пример https://searchlibrary.ohchr.org/record/23636

In [3]:
def fetch_xml(url):
    response = requests.get(url)
    if response.status_code == 200:
        print("OK: ", url)
        return response.text
    print(f"WRONG, {response.status_code}")
    if response.status_code == 202:
        sleep(2)
        print("!!!!!!!!!")
        print(response.text)
        return response.text
    else:
        print(f"Ошибка при загрузке URL {url}: статус-код {response.status_code}")
        return None

In [4]:
def get_filename(docid, path=f"{SAVEPATH}/pdfs"):
    return f"{path}/{docid.replace('/', '_')}.pdf"

In [5]:
def read_pdf(docid, onepage=True):
    filenamepath = get_filename(docid)
    if (not os.path.isfile(filenamepath)) or (not docid):
        return None
    with pdfplumber.open(filenamepath) as pdf:
        text = ""
        for page in pdf.pages:
            text += page.extract_text() + "\n\n"
            if onepage: return text
    return(text)

In [6]:
def get_sponsors(docid):
    if not docid: return None
    firstpage = read_pdf(docid)
    sponsors = None
    if firstpage:
        sponsors = extract_sponsor_countries_from_text(firstpage)
    print(f"Sponsors for {docid} are {sponsors}")
    return sponsors

In [7]:
def get_pdf(docid):
    #path = "../../data/raw/unhumanrights/pdfs"
    filename = get_filename(docid)
    if (os.path.isfile(filename)) or (not docid):
        print(f"File {filename} duplicated")
        return None
    r = requests.get(f"https://documents.un.org/api/symbol/access?s={docid}&l=en&t=pdf")
    #                   https://documents.un.org/api/symbol/access?s=E/CN.4/2003/L.42&l=en&t=pdf
    if r.status_code != 200:
        print(f"Error fetching {docid}: response code {r.status_code}")
        return None
    with open(filename, 'wb') as f:
        f.write(r.content)
        print(f"Wrote file {filename}")
    sleep(0.5)

In [8]:
def extract_sponsor_countries_from_text(page_text):
    # Удаляем разрывы строк внутри названий стран
    normalized_text = re.sub(r'-\s*\n\s*', '', page_text)  # Объединяем переносы
    normalized_text = re.sub(r'\s*\n\s*', ' ', normalized_text)  # Заменяем одиночные переносы на пробелы
    normalized_text = re.sub(r'\s+', ' ', normalized_text).strip()  # Удаляем лишние пробелы
    
    # Улучшенное регулярное выражение для поиска списка стран
    #   pattern = r'''
    #    ([A-Z][\w\s\',\-\.\*]+)  # Список стран (начинается с заглавной буквы)
    #    \s*:\s*                       # Двоеточие с возможными пробелами
    #    \*?                           # Опциональный символ * после двоеточия
    #    \s*                           # Возможные пробелы
    #    (?:revised\s+|updated\s+|edited\s+)?  # Опциональные модификаторы
    #    (?:draft\s+)                 # Слово "draft"
    #    resolution  # Тип документа
    #'''
    pattern = r'([A-Z][\w\s\'\’\,\-\.\*]+)\s*:\s*\*?(?:revised\s+|updated\s+|edited\s+)?draft resolution'
    #r'([A-Z][\w\s\',\-\.\*]+)\s*:\s*draft resolution'
    match = re.search(pattern, normalized_text)
    
    if not match:
        return []
    
    countries_str = match.group(1).strip()
    
    # Обработка случая с союзом "and" между последними странами
    if " and " in countries_str:
        # Разделяем на части перед "and" и после
        parts = re.split(r'\s+and\s+', countries_str)
        # Последний элемент - последняя страна
        last_country = parts[-1]
        # Все что перед "and" - это предыдущие страны
        first_part = ' and '.join(parts[:-1])
        # Собираем страны из первой части (разделенные запятыми) и добавляем последнюю
        countries_list = [c.strip() for c in first_part.split(',')] + [last_country.strip()]
    else:
        countries_list = [c.strip() for c in countries_str.split(',')]
    
    # Очистка названий стран
    countries = []
    for country in countries_list:
        # Удаляем примечания в скобках
        country = re.sub(r'\s*\([^)]*\)', '', country)
        # Удаляем спецсимволы в конце названия
        country = re.sub(r'[\*\.,\d]+$', '', country).strip()
        # Удаляем начальные/конечные небуквенные символы
        country = re.sub(r'^\W+|\W+$', '', country)
            
    return countries

def OLDextract_sponsor_countries_from_text(page_text):
    # Нормализуем пробелы для упрощения обработки
    normalized_text = re.sub(r'\s+', ' ', page_text).strip()
    
    # Ищем паттерн: страны + ": draft resolution"
    # Улучшенный паттерн для захвата сложных названий
    #pattern = r'([A-Z][\w\s\',\-\.\(\)\*]+):\s*draft resolution'
    pattern = r'''
        ([A-Z][\w\s\',\-\.\(\)\*]+)  # Список стран (начинается с заглавной буквы)
        \s*:\s*                       # Двоеточие с возможными пробелами
        \*?                           # Опциональный символ * после двоеточия
        \s*                           # Возможные пробелы
        (?:revised\s+|updated\s+|edited\s+)?  # Опциональные модификаторы
        (?:draft\s+)                 # слово "draft"
        (?:resolution|decision|proposal)  # Тип документа
    '''
    
    match = re.search(pattern, normalized_text)
    
    if not match:
        return []
    
    countries_str = match.group(1).strip()
    
    # Разделяем страны, очищаем от лишних символов
    countries = []
    for raw_country in countries_str.split(','):
        country = raw_country.strip()
        
        # Удаляем примечания в скобках (если есть)
        country = re.sub(r'\s*\([^)]*\)', '', country)
        
        # Удаляем специальные символы в конце названия
        country = re.sub(r'[\*\.,\d]+$', '', country).strip()
        
        # Пропускаем пустые элементы и не-строки
        if country and re.search(r'[a-zA-Z]', country):
            countries.append(country)
    
    return countries

In [9]:
def humanunxml_to_df(xml_content):
    # Парсинг XML
    NS = {'marc': 'http://www.loc.gov/MARC21/slim'}
    root = ET.fromstring(xml_content)
    
    # Создаем структуры для хранения данных в реляционном формате
    records = []
    subfields = []
    links = []
    sponsors = []
    
    for i, record in enumerate(root.findall('marc:record', NS), 1):
        # Основная информация о записи
        record_id = record.find('marc:controlfield[@tag="001"]', NS).text
        control_005 = record.find('marc:controlfield[@tag="005"]', NS).text
        
        # Извлекаем заголовок (поле 245)
        title_field = record.find('marc:datafield[@tag="245"]', NS)
        title = " ".join([sf.text for sf in title_field.findall('marc:subfield', NS) if sf.text]) if title_field else None
        
        # Извлекаем дату (поле 269)
        date_field = record.find('marc:datafield[@tag="269"]', NS)
        date = date_field.find('marc:subfield[@code="a"]', NS).text if date_field else None
        
        # Извлекаем спонсоров (поле 500)
        sponsor_field = record.find('marc:datafield[@tag="500"]', NS)
        sponsor = sponsor_field.find('marc:subfield[@code="a"]', NS).text if sponsor_field else None
        
        # Извлекаем спонсоров (поле 500)
        dateold_field = record.find('marc:datafield[@tag="992"]', NS)
        dateold = dateold_field.find('marc:subfield[@code="a"]', NS).text if dateold_field else None
        
        # Добавляем основную запись
        records.append({
            'record_id': record_id,
            'control_005': control_005,
            'title': title,
            'date': date,
            'dateold' : dateold,
            'sponsors': sponsor
        })
        
        # Обрабатываем все datafields
        for df in record.findall('marc:datafield', NS):
            tag = df.get('tag')
            for sf in df.findall('marc:subfield', NS):
                subfields.append({
                    'record_id': record_id,
                    'field_tag': tag,
                    'subfield_code': sf.get('code'),
                    'value': sf.text
                })
        
        # Обрабатываем ссылки отдельно (поле 856)
        for link in record.findall('marc:datafield[@tag="856"]', NS):
            try:
                url = link.find('marc:subfield[@code="u"]', NS).text
                desc = link.find('marc:subfield[@code="y"]', NS).text if link.find('marc:subfield[@code="y"]', NS) is not None else None
                links.append({
                    'record_id': record_id,
                    'url': url,
                    'description': desc
                })
            except Exception:
                print("No links")
    
    # Создаем DataFrame'ы
    df_records = pd.DataFrame(records)
    df_subfields = pd.DataFrame(subfields)
    df_links = pd.DataFrame(links)
    
    # Функция для извлечения конкретных полей в основные записи
    def extract_field_to_main(df, tag, code, new_column):
        values = df[(df['field_tag'] == tag) & (df['subfield_code'] == code)]
        for _, row in values.iterrows():
            idx = df_records[df_records['record_id'] == row['record_id']].index
            if not idx.empty:
                df_records.loc[idx[0], new_column] = row['value']
    
    # Извлекаем важные поля в основную таблицу
    extract_field_to_main(df_subfields, '591', 'a', 'vote_status')
    extract_field_to_main(df_subfields, '791', 'a', 'resolution_number')
    extract_field_to_main(df_subfields, '793', 'v', 'meeting_type')
    extract_field_to_main(df_subfields, '952', 'a', 'adoption_info')
    extract_field_to_main(df_subfields, '981', 'a', 'council')
    extract_field_to_main(df_subfields, '991', 'd', 'topic')
    extract_field_to_main(df_subfields, '992', 'd', 'date_old')
    
    # Результат
    #print("Основная таблица записей:")
    #print(df_records.head())
    
    #print("\nТаблица подполей:")
    #print(df_subfields.head())
    
    #print("\nТаблица ссылок:")
    #print(df_links.head())
    return df_records, df_links

In [10]:
def get_draftid(url):
    
    prefixes = [
        "https://docs.un.org/en/",
        "http://docs.un.org/en/",
        "https://www.docs.un.org/en/",
        "http://www.docs.un.org/en/",
        "https://undocs.org/",
        "http://undocs.org/",
        "http://www.undocs.org",
        "https://www.undocs.org"
    ]
    for prefix in prefixes:
        if url.startswith(prefix) and ("/L." in url):
            return url[len(prefix):]
    return None  # Обработка несоответствующего URL
    
#get_draftid("https://docs.un.org/en/L.10")

In [11]:
def convert_to_datetime(date_str):
    """
    Преобразует строку с датой в формате YYYYMMDD или YYYY-MM-DD в datetime
    """
    if pd.isna(date_str):
        return None
    
    # Удаляем возможные лишние символы
    date_str = str(date_str).strip()
    
    try:
        # Пробуем разные форматы дат
        if '-' in date_str:
            return datetime.strptime(date_str, '%Y-%m-%d')
        elif len(date_str) == 8 and date_str.isdigit():
            return datetime.strptime(date_str, '%Y%m%d')
        else:
            # Для других форматов можно добавить дополнительные условия
            return None
    except (ValueError, TypeError):
        return None

In [12]:
countrylabels = list(set(loadKeyDf(load_extra=True).loc[:,:'STATE_en_historic_2'].fillna('Канада').values.ravel()))
# Удаляем пустые строки и сортируем по длине (для корректного поиска подстрок)
list_countrylabels = sorted([b for b in countrylabels if b], key=len, reverse=True)
# Создаем регулярное выражение
pattern = re.compile("|".join(map(re.escape, list_countrylabels)))

def match_countries(potential_countrylist):
    results = []
    if not potential_countrylist: return None
    for country_candidate in potential_countrylist:
        # Находим все уникальные совпадения
        matches = set(pattern.findall(country_candidate))
        # Фильтруем только элементы из исходного list_b
        valid_matches = [b for b in list_countrylabels if b in matches]
        if len(valid_matches)==0:
            continue
        results += valid_matches
    return results

def match_countries_nolist(potential_countrylist):
    results = []
    if not potential_countrylist: return None
    # Находим все уникальные совпадения
    matches = set(pattern.findall(potential_countrylist))
    # Фильтруем только элементы из исходного list_b
    valid_matches = [b for b in list_countrylabels if b in matches]
    return valid_matches

In [13]:
def removeslash(x):
    if x:
        if x[0] == '/' and len(x)>1:
            return x[1:]
    return x

def printen(x):
    if not x:
        return x
    if len(x) <= 3:
        return x
    if x[0:3].lower() == 'en/':
        print(x)
    return x

def removeen(x):
    if not x:
        return x
    if len(x) <= 3:
        return x
    if x[0:3].lower() == 'en/':
        return x[3:]
    return x
            
def printifslash(x):
    if x:
        if x[0] == '/':
            print(x)
            
#df['draft_id'] = df['draft_id'].apply(removeslash)
#df['draft_id'] = df['draft_id'].apply(removeen)
#df['draft_id'].apply(printifslash)
#df['draft_id'].apply(printen)


In [14]:
if __name__ == "__main__":
    dfs = []
    dfs_links = []
    for i in range(0, 100):
        #1987
        #url=f"https://searchlibrary.ohchr.org/search?cc=Voting&ln=en&fct__4=1987&of=xm&jrec={1 + i * 200}&rg=200"
        url = f"https://searchlibrary.ohchr.org/search?cc=Voting&ln=en&p=&f=&action_search=Search&rm=&sf=&so=d&c=Voting&c=&fct__4=1985&fct__4=1986&fct__4=1987&c=&fct__4=1988&fct__4=1989&fct__4=1990&fct__4=1991&fct__4=1992&of=xm&jrec={1 + i * 200}&rg=200"
        #url = f"https://searchlibrary.ohchr.org/search?cc=Voting&ln=en&p=&f=&action_search=Search&rm=&sf=&so=d&c=Voting&c=&of=xm&jrec={1 + i * 200}&rg=200"
               #"https://searchlibrary.ohchr.org/search?cc=Voting&ln=en&p=&f=&action_search=Search&rm=&sf=&so=d&rg=200&c=Voting&c=&of=ht&fct__4=1991&fct__4=1990&fct__4=1989&fct__4=1988&fct__4=1987&fct__4=1986&fct__4=1985"
        xml_content = fetch_xml(url)
        if xml_content: 
            df, df_links = humanunxml_to_df(xml_content)
            dfs += df,
            dfs_links += df_links,
        else:
            break
        if df.shape[0]<200:
            print(f"Out of options to fetch, last batch is:{df.shape[0]}")
            break
        sleep(1)
    

OK:  https://searchlibrary.ohchr.org/search?cc=Voting&ln=en&p=&f=&action_search=Search&rm=&sf=&so=d&c=Voting&c=&fct__4=1985&fct__4=1986&fct__4=1987&c=&fct__4=1988&fct__4=1989&fct__4=1990&fct__4=1991&fct__4=1992&of=xm&jrec=1&rg=200
OK:  https://searchlibrary.ohchr.org/search?cc=Voting&ln=en&p=&f=&action_search=Search&rm=&sf=&so=d&c=Voting&c=&fct__4=1985&fct__4=1986&fct__4=1987&c=&fct__4=1988&fct__4=1989&fct__4=1990&fct__4=1991&fct__4=1992&of=xm&jrec=201&rg=200
OK:  https://searchlibrary.ohchr.org/search?cc=Voting&ln=en&p=&f=&action_search=Search&rm=&sf=&so=d&c=Voting&c=&fct__4=1985&fct__4=1986&fct__4=1987&c=&fct__4=1988&fct__4=1989&fct__4=1990&fct__4=1991&fct__4=1992&of=xm&jrec=401&rg=200
OK:  https://searchlibrary.ohchr.org/search?cc=Voting&ln=en&p=&f=&action_search=Search&rm=&sf=&so=d&c=Voting&c=&fct__4=1985&fct__4=1986&fct__4=1987&c=&fct__4=1988&fct__4=1989&fct__4=1990&fct__4=1991&fct__4=1992&of=xm&jrec=601&rg=200
Out of options to fetch, last batch is:91


In [15]:
df = pd.concat(dfs)
df_links = pd.concat(dfs_links)
df.drop_duplicates(inplace=True)
df['date']=df['date'].apply(convert_to_datetime)
df_links = df_links.dropna()[(df_links.dropna()['description'].str.contains("raf"))|df_links.dropna()['description'].str.contains("Sessional report")].drop_duplicates()
df_nocorr = df_links[~df_links['description'].str.contains("Corrigendum")]
df_corr = df_links[df_links['description'].str.contains("Corrigendum")]
df = df.merge(df_nocorr, left_on = 'record_id', right_on='record_id')
df['draft_id'] = df['url'].apply(get_draftid)
df['draft_id'] = df['draft_id'].apply(removeslash) # Some ids include / or en/, which impedes download
df['draft_id'] = df['draft_id'].apply(removeen)
# TEMPORARY, REMOVE COMMENT IF DOING OVER
#df['draft_id'].dropna().apply(get_pdf)
df['year'] = df['date'].dt.year

In [18]:
print(df.shape)

(726, 16)


In [17]:
df = df[(df['description']!='Sessional report')|(df['title'].str.contains('resolution'))]

In [19]:
df['sponsors_maybe'] = df['draft_id'].apply(get_sponsors)
df['sponsors_new']=df['sponsors'].apply(match_countries_nolist)
#df['sponsors_new']=df['sponsors_maybe'].apply(match_countries)
df['affected'] = df['title'].apply(match_countries_nolist)

Sponsors for E/CN.4/1992/S-2/L.2 are []
Sponsors for zh/E/CN.4/1992/L.65 are None
Sponsors for zh/E/CN.4/1992/L.60 are None
Sponsors for zh/E/CN.4/1992/L.64 are None
Sponsors for zh/E/CN.4/1992/L.88 are None
Sponsors for zh/E/CN.4/1992/L.82/Rev.1 are None
Sponsors for zh/E/CN.4/1992/L.84 are None
Sponsors for zh/E/CN.4/1992/L.86 are None
Sponsors for zh/E/CN.4/1992/L.63 are None
Sponsors for zh/E/CN.4/1992/L.71 are None
Sponsors for zh/E/CN.4/1992/L.77 are None
Sponsors for zh/E/CN.4/1992/L.76 are None
Sponsors for zh/E/CN.4/1992/L.62/Rev.1 are None
Sponsors for zh/E/CN.4/1992/L.69 are None
Sponsors for zh/E/CN.4/1992/L.24 are None
Sponsors for zh/E/CN.4/1992/L.72 are None
Sponsors for zh/E/CN.4/1992/L.61 are None
Sponsors for zh/E/CN.4/1992/L.46 are None
Sponsors for zh/E/CN.4/1992/L.53 are None
Sponsors for zh/E/CN.4/1992/L.54 are None
Sponsors for zh/E/CN.4/1992/L.58 are None
Sponsors for zh/E/CN.4/1992/L.51 are None
Sponsors for zh/E/CN.4/1992/L.52 are None
Sponsors for zh/E/CN.4/1

In [22]:
df.groupby('year').count()

,record_id,control_005,title,date,dateold,sponsors,vote_status,resolution_number,meeting_type,adoption_info,council,topic,url,description,draft_id,sponsors_maybe,sponsors_new,affected
year,,,,,,,,,,,,,,,,,,
1985,62,62,62,62,62,0,62,62,62,62,62,60,62,62,1,1,0,62
1986,70,70,70,70,70,0,70,70,70,70,70,69,70,70,0,0,0,70
1987,63,63,63,63,63,0,63,63,63,63,63,62,63,63,0,0,0,63
1988,86,86,86,86,86,0,86,86,86,86,86,86,86,86,0,0,0,86
1989,86,86,86,86,86,0,86,86,86,86,86,86,86,86,0,0,0,86
1990,91,91,91,91,91,0,91,91,91,91,91,91,91,91,0,0,0,91
1991,88,88,88,88,88,0,88,88,88,88,88,88,88,88,0,0,0,88
1992,180,180,180,180,180,0,180,180,180,180,180,178,180,180,77,1,0,180


In [21]:
mask = df['sponsors'].isna()
df.loc[mask, 'sponsors_new'] = df.loc[mask, 'sponsors_maybe'].apply(match_countries)

In [671]:
#df['sponsors'].fillna([None], inplace=True)
df=df.explode('sponsors_new', ignore_index=True)

In [672]:
#df['affected'].fillna([None], inplace=True)
df=df.explode('affected', ignore_index=True)

In [23]:
df

,record_id,control_005,title,date,dateold,sponsors,vote_status,resolution_number,meeting_type,adoption_info,council,topic,url,description,draft_id,year,sponsors_maybe,sponsors_new,affected
0,21027,20240731103050.0,The situation of human rights in the territory...,1992-12-01,19921201,None,RECORDED,E/CN.4/RES/1992/S-2/1,PLENARY MEETING,E/CN.4/1992/S-2/SR.4,Commission on Human Rights,YUGOSLAVIA,http://undocs.org/E/1992/22/Add.2(SUPP),Sessional report,None,1992,None,None,[Yugoslavia]
1,21027,20240731103050.0,The situation of human rights in the territory...,1992-12-01,19921201,None,RECORDED,E/CN.4/RES/1992/S-2/1,PLENARY MEETING,E/CN.4/1992/S-2/SR.4,Commission on Human Rights,YUGOSLAVIA,http://undocs.org/E/CN.4/1992/S-2/L.2,Draft resolution,E/CN.4/1992/S-2/L.2,1992,[],None,[Yugoslavia]
2,21030,20200129195230.0,The situation of human rights in the territory...,1992-08-14,19920814,None,ADOPTED WITHOUT VOTE,E/CN.4/RES/1992/S-1/1,PLENARY MEETING,E/CN.4/1992/S-1/SR.4,Commission on Human Rights,YUGOSLAVIA,http://undocs.org/E/1992/22/Add.1/Rev.1,Sessional report,None,1992,None,None,[Yugoslavia]
4,23081,20200129195306.0,Rationalization of the work of the Commission ...,1992-03-06,19920306,None,ADOPTED WITHOUT VOTE,E/CN.4/RES/1992/83,PLENARY MEETING,E/CN.4/1992/SR.57,Commission on Human Rights,NaN,http://undocs.org/E/1992/22(SUPP),Sessional report,None,1992,None,None,[]
6,23072,20200129195306.0,Implementation of the Convention on the Rights...,1992-03-05,19920305,None,ADOPTED WITHOUT VOTE,E/CN.4/RES/1992/75,PLENARY MEETING,E/CN.4/1992/SR.55,Commission on Human Rights,RIGHTS OF THE CHILD,http://undocs.org/E/1992/22(SUPP),Sessional report,None,1992,None,None,[]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
816,24017,20240730135756.0,The adverse consequences for the enjoyment of ...,1985-02-26,19850226,None,RECORDED,E/CN.4/RES/1985/9,PLENARY MEETING,E/CN.4/1985/SR.32,Commission on Human Rights,SOUTHERN AFRICA,http://undocs.org/E/1985/22(supp),Sessional report,None,1985,None,None,[]
817,24018,20240730135736.0,Implementation of the International Convention...,1985-02-26,19850226,None,RECORDED,E/CN.4/RES/1985/10,PLENARY MEETING,E/CN.4/1985/SR.32,Commission on Human Rights,APARTHEID - TREATY COMPLIANCE,http://undocs.org/E/1985/22(supp),Sessional report,None,1985,None,None,[]
818,24009,20240730104129.0,Question of the violation of human rights in t...,1985-02-19,19850219,None,RECORDED,E/CN.4/RES/1985/1B,PLENARY MEETING,E/CN.4/1985/SR.21,Commission on Human Rights,ISRAEL - ARAB COUNTRIES,http://undocs.org/E/1985/22(supp),Sessional report,None,1985,None,None,[Palestine]
819,24010,20240730104109.0,Human rights situation on occupied Syrian terr...,1985-02-19,19850219,None,RECORDED,E/CN.4/RES/1985/2,PLENARY MEETING,E/CN.4/1985/SR.21,Commission on Human Rights,SYRIAN ARAB REPUBLIC,http://undocs.org/E/1985/22(supp),Sessional report,None,1985,None,None,[Syria]


In [24]:
df.to_csv(f"{SAVEPATH}/humanun_old.csv") # pre 1993 resolutions with buggy sponsorship
#df.to_csv(f"{SAVEPATH}/humanun.csv")

In [604]:
df[df['sponsors_new']=='Norway'].head(50)

,record_id,control_005,title,date,dateold,sponsors,vote_status,resolution_number,meeting_type,adoption_info,council,topic,url,description,draft_id,year,sponsors_maybe,sponsors_new,affected
28,31249,20231109102822.0,Responding to the human rights and humanitaria...,2023-10-11,20231011,Main sponsors : United Kingdom of Great Britai...,RECORDED,A/HRC/RES/54/2,PLENARY MEETING,Adopted at the 46th meeting,Human Rights Council,HUMAN RIGHTS IN ARMED CONFLICTS,http://undocs.org/A/HRC/54/L.18,Draft resolution,A/HRC/54/L.18,2023,[October 2023 Agenda item 2 Annual report of t...,Norway,Sudan
211,30372,20230621161800.0,Mandate of Special Rapporteur on the situation...,2023-04-03,20230403,Main sponsors : Norway,ADOPTED WITHOUT VOTE,A/HRC/RES/52/4,PLENARY MEETING,Adopted at the 55th meeting,Human Rights Council,HUMAN RIGHTS DEFENDERS - SPECIAL RAPPORTEURS,http://undocs.org/A/HRC/52/L.1,Draft resolution,A/HRC/52/L.1,2023,[April 2023 Agenda item 3 Promotion and protec...,Norway,NaN
241,30357,20230621145332.0,Advancing human rights in South Sudan : resolu...,2023-04-03,20230403,Main sponsors : United Kingdom of Great Britai...,RECORDED,A/HRC/RES/52/1,PLENARY MEETING,Adopted at the 55th meeting,Human Rights Council,SOUTH SUDAN,http://undocs.org/A/HRC/52/L.27,Draft resolution,A/HRC/52/L.27,2023,[April 2023 Agenda item 2 Annual report of the...,Norway,South Sudan
285,30389,20230623093241.0,"Human rights, democracy and the rule of law : ...",2023-04-03,20230403,"Main sponsors : Romania, Morocco, Norway, Peru...",ADOPTED WITHOUT VOTE,A/HRC/RES/52/22,PLENARY MEETING,Adopted at the 55th meeting,Human Rights Council,DEMOCRACY AND HUMAN RIGHTS - RULE OF LAW,http://undocs.org/A/HRC/52/L.39,Draft resolution,A/HRC/52/L.39,2023,[April 2023 Agenda item 3 Promotion and protec...,Norway,NaN
306,30319,20230306155346.0,Strengthening the voluntary funds for the univ...,2022-10-07,20221007,"Main sponsors : Argentina, Armenia, Fiji, Norw...",ADOPTED WITHOUT VOTE,A/HRC/RES/51/30,PLENARY MEETING,Adopted at the 44th meeting,Human Rights Council,UN. Human Rights Council. Universal Periodic R...,http://undocs.org/A/HRC/51/L.10/Rev.1,Draft resolution,A/HRC/51/L.10/Rev.1,2022,[October 2022 Agenda item 6 Universal periodic...,Norway,NaN
314,30322,20230306160141.0,Enhancement of technical cooperation and capac...,2022-10-07,20221007,"Main sponsors : Thailand, Brazil, Honduras, In...",ADOPTED WITHOUT VOTE,A/HRC/RES/51/34,PLENARY MEETING,Adopted at the 44th meeting,Human Rights Council,TECHNICAL COOPERATION - CAPACITY BUILDING,http://undocs.org/A/HRC/51/L.15/Rev.1,Draft resolution,A/HRC/51/L.15/Rev.1,2022,[October 2022 Agenda item 10 Technical assista...,Norway,NaN
441,30313,20230303154034.0,Human rights and Indigenous Peoples : resoluti...,2022-10-06,20221006,None,ADOPTED WITHOUT VOTE,A/HRC/RES/51/18,PLENARY MEETING,Adopted at the 41st meeting,Human Rights Council,INDIGENOUS PEOPLES,http://undocs.org/A/HRC/51/L.39,Draft resolution,A/HRC/51/L.39,2022,[October 2022 Agenda item 3 Promotion and prot...,Norway,NaN
491,30260,20221125091209.0,Situation of human rights of women and girls i...,2022-07-08,20220708,None,ADOPTED WITHOUT VOTE,A/HRC/RES/50/14,PLENARY MEETING,Adopted at the 41st meeting,Human Rights Council,WOMEN - GIRLS - AFGHANISTAN,http://undocs.org/A/HRC/50/L.62,Draft resolution,A/HRC/50/L.62,2022,[July 2022 Agenda item 1 Organizational and pr...,Norway,Afghanistan
543,30261,20221125094052.0,Freedom of opinion and expression : resolution...,2022-07-08,20220708,None,ADOPTED WITHOUT VOTE,A/HRC/RES/50/15,PLENARY MEETING,Adopted at the 41st meeting,Human Rights Council,FREEDOM OF OPINION AND EXPRESSION,http://undocs.org/A/HRC/50/L.11,Draft resolution,A/HRC/50/L.11,2022,[July 2022 Agenda item 3 Promotion and protect...,Norway,NaN
597,30264,20221125095023.0,Elimination of all forms of discrimination aga...,2022-07-08,20220708,None,ADOPTED WITHOUT VOTE,A/HRC/RES/50/18,PLENARY MEETING,Adopted at the 41st meeting,Human Rights Council,GENDER DISCRIMINATION,http://undocs.org/A/HR

In [591]:
match_countries(df[df['record_id']=='30294'].loc[1779]['sponsors_maybe'])

KeyError: 1779